In [1]:
# mtx2adata.ipynb - convert sparse matrix file into adata file.

In [2]:
import os
import pandas as pd
from sccnasim.xlib.xbarcode import rand_cell_barcodes
from sccnasim.xlib.xdata import load_10x_data

In [3]:
data_dir = "/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/simulator_evaluation/base/gen_data/afc_scdesign2_ind/raw_matrix"
gene_anno_fn = "/groups/cgsd/xianjie/projects/cna-benchmark/HCC3N_600spot/base/data/gene_anno/refdata-cellranger-GRCh38-3.0.0.valid_name.uniq_name.chrom1-22XY.5column.genes.tsv"
n_spots = 1200

out_dir = data_dir
out_adata_fn = os.path.join(out_dir, "scDesign2_ind.raw_matrix.h5ad")

## Generate random barcodes

In [4]:
#random.seed(1)
barcodes = rand_cell_barcodes(m = 16, n = n_spots, suffix = '-1', sort = False)
print(len(barcodes))
barcodes[:10]

1200


['CGTGGGCCCACGCGCC-1',
 'CCCACTGCACTTAAGA-1',
 'CCGCAAGCTTGAGATT-1',
 'CGCCACCCGAACCTTC-1',
 'CTTACCAGGTCATCTT-1',
 'ACATATTTCATCTGAA-1',
 'AATGTGCGTTCTGACG-1',
 'ACGGACTTGACCCCCA-1',
 'CGTTTAGGTCATATGT-1',
 'CCAACTCAAGGGAGCC-1']

In [5]:
n = int(n_spots / 2)
cell_anno = pd.DataFrame(data = dict(
    cell = barcodes,
    cell_type = ['normal'] * n + ['tumor'] * n
))
cell_anno

,cell,cell_type
0,CGTGGGCCCACGCGCC-1,normal
1,CCCACTGCACTTAAGA-1,normal
2,CCGCAAGCTTGAGATT-1,normal
3,CGCCACCCGAACCTTC-1,normal
4,CTTACCAGGTCATCTT-1,normal
...,...,...
1195,GAGAGAAACAGGATCA-1,tumor
1196,GCTGCGAGCCCAGACA-1,tumor
1197,GCTAGGAGCCCATTCC-1,tumor
1198,ATAAAAGAATGCGAGT-1,tumor


In [6]:
cell_anno = cell_anno.sort_values(by = ['cell'])
cell_anno

,cell,cell_type
273,AAAAAAATAAAGGATC-1,normal
363,AAAAACGAAGGGCTCA-1,normal
720,AAAAGTGACATAAGGT-1,tumor
646,AAAATCCCACTGTCCC-1,tumor
1078,AAACCCGATTTAGGAA-1,tumor
...,...,...
57,TTTCTTCTGGAAGCTC-1,normal
223,TTTGTCATCAAGCCGC-1,normal
398,TTTTACCCCGTTTTTA-1,normal
1153,TTTTCGCTTCTGAATT-1,tumor


In [7]:
cell_anno['cell_type'].value_counts()

cell_type
normal    600
tumor     600
Name: count, dtype: int64

In [8]:
cell_anno[['cell']].to_csv(
    os.path.join(out_dir, 'barcodes.tsv'),
    header = False,
    index = False
)

In [9]:
cell_anno.to_csv(
    os.path.join(out_dir, 'spot_anno.tsv'),
    header = False,
    index = False,
    sep = '\t'
)

## Construct adata

In [10]:
adata = load_10x_data(data_dir)
adata

/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/xianjie/.anaconda3/envs/SCSC/lib/python3.11/site-packages/anndata-0.10.7-py3.11.egg/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


AnnData object with n_obs × n_vars = 1200 × 32286
    obs: 'cell'
    var: 'feature_id', 'feature_name'

In [11]:
adata.obs = adata.obs.merge(cell_anno, on = 'cell', how = 'left')
adata.obs

,cell,cell_type
0,AAAAAAATAAAGGATC-1,normal
1,AAAAACGAAGGGCTCA-1,normal
2,AAAAGTGACATAAGGT-1,tumor
3,AAAATCCCACTGTCCC-1,tumor
4,AAACCCGATTTAGGAA-1,tumor
...,...,...
1195,TTTCTTCTGGAAGCTC-1,normal
1196,TTTGTCATCAAGCCGC-1,normal
1197,TTTTACCCCGTTTTTA-1,normal
1198,TTTTCGCTTCTGAATT-1,tumor


In [12]:
adata.var.columns = ['gene_id', 'gene']
adata.var

,gene_id,gene
0,ENSG00000243485,MIR1302-2HG
1,ENSG00000237613,FAM138A
2,ENSG00000186092,OR4F5
3,ENSG00000238009,AL627309.1
4,ENSG00000239945,AL627309.3
...,...,...
32281,ENSG00000203993,ARRDC1-AS1
32282,ENSG00000181090,EHMT1
32283,ENSG00000203987,AL772363.1
32284,ENSG00000148408,CACNA1B


In [13]:
gene_anno = pd.read_csv(gene_anno_fn, sep = '\t', header = None, dtype = {0:str})
gene_anno.columns = ['chrom', 'start', 'end', 'gene', 'strand']
gene_anno

,chrom,start,end,gene,strand
0,1,29554,31109,MIR1302-2HG,+
1,1,34554,36081,FAM138A,-
2,1,65419,71585,OR4F5,+
3,1,89295,133723,AL627309.1,-
4,1,89551,91105,AL627309.3,-
...,...,...,...,...,...
33467,Y,25063083,25099892,TTTY4C,-
33468,Y,25183643,25184773,TTTY17C,-
33469,Y,25378300,25394719,LINC00266-4P,-
33470,Y,25622162,25624902,CDY1,+


In [14]:
adata.var = adata.var.merge(gene_anno, on = 'gene', how = 'left')
adata.var

,gene_id,gene,chrom,start,end,strand
0,ENSG00000243485,MIR1302-2HG,1,29554,31109,+
1,ENSG00000237613,FAM138A,1,34554,36081,-
2,ENSG00000186092,OR4F5,1,65419,71585,+
3,ENSG00000238009,AL627309.1,1,89295,133723,-
4,ENSG00000239945,AL627309.3,1,89551,91105,-
...,...,...,...,...,...,...
32281,ENSG00000203993,ARRDC1-AS1,9,137615332,137618906,-
32282,ENSG00000181090,EHMT1,9,137618963,137870016,+
32283,ENSG00000203987,AL772363.1,9,137867925,137892570,-
32284,ENSG00000148408,CACNA1B,9,137877789,138124624,+


## Save adata

In [15]:
adata.X = adata.X.toarray()

In [16]:
adata

AnnData object with n_obs × n_vars = 1200 × 32286
    obs: 'cell', 'cell_type'
    var: 'gene_id', 'gene', 'chrom', 'start', 'end', 'strand'

In [17]:
adata.write(out_adata_fn, compression = 'gzip')